# 02 — Before vs After Training: What Does Training Actually Do?

This notebook loads two copies of Qwen 2.5 0.5B:
1. **Trained**: the real pretrained model (learned from trillions of tokens)
2. **Random**: same architecture, but freshly initialized with random weights

By comparing them, you'll see exactly what training changes — and why the untrained model produces gibberish.

**Note**: This notebook uses raw HuggingFace (not TransformerLens) because we need `from_config()` for random initialization.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from utils.model_loading import load_hf_model, load_random_model, get_tokenizer, format_param_count
from utils.visualization import apply_theme, plot_distribution_comparison, plot_svd_spectrum, ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED, ACCENT_PURPLE, ACCENT_TEAL, TEXT_COLOR, DARK_BG, DARK_SURFACE, PALETTE, _style_box
apply_theme()

# Load both models
trained = load_hf_model("0.5b", device="cpu")
random_model = load_random_model("0.5b", device="cpu")
tokenizer = get_tokenizer("0.5b")

trained.eval()
random_model.eval()
print("Both models loaded on CPU.")

## 1. Generation: The Visceral Difference

Let's feed the same prompt to both models and see what comes out.

In [ ]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    trained_out = trained.generate(**inputs, max_new_tokens=30, do_sample=False)
    random_out = random_model.generate(**inputs, max_new_tokens=30, do_sample=True, temperature=1.0)

print("PROMPT:", prompt)
print()
print("TRAINED MODEL:")
print(tokenizer.decode(trained_out[0], skip_special_tokens=True))
print()
print("RANDOM MODEL (untrained):")
print(tokenizer.decode(random_out[0], skip_special_tokens=True))
print()
print("The random model is rolling a ~152K-sided die per token. No knowledge, no grammar, no coherence.")

## 2. Weight Distributions: Random Gaussian → Structured

The random model initializes all weights from `Normal(0, 0.02)`. Training reshapes these distributions dramatically.

In [ ]:
# Compare weight distributions for key layers
pairs = [
    ("model.embed_tokens.weight", "Token Embedding"),
    ("model.layers.0.self_attn.q_proj.weight", "Layer 0: Q Projection"),
    ("model.layers.12.mlp.gate_proj.weight", "Layer 12: MLP Gate"),
    ("model.layers.23.self_attn.o_proj.weight", "Last Layer: Attention Output"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

trained_params = dict(trained.named_parameters())
random_params = dict(random_model.named_parameters())

for idx, (name, title) in enumerate(pairs):
    plot_distribution_comparison(
        random_params[name].data, trained_params[name].data,
        label_a="Random Init", label_b="Trained",
        title=title, bins=150, ax=axes[idx]
    )
    # Add std annotation
    r_std = random_params[name].data.float().std().item()
    t_std = trained_params[name].data.float().std().item()
    axes[idx].text(0.98, 0.85, f"σ_random={r_std:.4f}\nσ_trained={t_std:.4f}",
                   transform=axes[idx].transAxes, ha="right", fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()
print("Training dramatically widens and reshapes weight distributions.")

## 3. Frobenius Norms: How "Large" Are the Weights?

The Frobenius norm (||W||_F = sqrt(sum of squared elements)) measures the overall magnitude of a weight matrix.

In [ ]:
# Compute Frobenius norms for all weight matrices in transformer layers
trained_norms = []
random_norms = []
labels = []

for i in range(24):  # 24 layers
    for component in ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", 
                       "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"]:
        name = f"model.layers.{i}.{component}.weight"
        trained_norms.append(trained_params[name].data.float().norm().item())
        random_norms.append(random_params[name].data.float().norm().item())
        if i == 0:
            labels.append(component.split(".")[-1])

# Plot per-layer (average across components within each layer)
n_components = 7
trained_by_layer = [np.mean(trained_norms[i*n_components:(i+1)*n_components]) for i in range(24)]
random_by_layer = [np.mean(random_norms[i*n_components:(i+1)*n_components]) for i in range(24)]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(24)
width = 0.35
ax.bar(x - width/2, random_by_layer, width, label="Random Init", color=ACCENT_RED, alpha=0.7)
ax.bar(x + width/2, trained_by_layer, width, label="Trained", color=ACCENT_BLUE, alpha=0.7)
ax.set_xlabel("Layer")
ax.set_ylabel("Mean Frobenius Norm")
ax.set_title("Weight Magnitude by Layer: Random vs Trained")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Random init norms are uniform (~{np.mean(random_by_layer):.1f})")
print(f"Trained norms vary by layer (range: {min(trained_by_layer):.1f} to {max(trained_by_layer):.1f})")
print("The model develops structure — some layers have higher-norm weights than others.")

## 4. Singular Value Spectrum: The Emergence of Low-Rank Structure

The singular value decomposition (SVD) reveals the "effective rank" of a matrix. Random matrices have flat spectra (all singular values roughly equal). Trained matrices develop heavy-tailed spectra — a few dominant directions capture most of the information.

This is why techniques like **LoRA** (Low-Rank Adaptation) work: trained weights are approximately low-rank.

In [ ]:
# SVD comparison for embedding matrix and an attention matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

svd_pairs = [
    ("model.embed_tokens.weight", "Token Embedding"),
    ("model.layers.12.self_attn.q_proj.weight", "Layer 12: Q Projection"),
]

for idx, (name, title) in enumerate(svd_pairs):
    ax = axes[idx]
    
    # Compute SVD for both
    t_data = trained_params[name].data.float()
    r_data = random_params[name].data.float()
    
    t_svd = torch.linalg.svdvals(t_data)[:100].numpy()
    r_svd = torch.linalg.svdvals(r_data)[:100].numpy()
    
    ax.plot(r_svd, label="Random Init", color=ACCENT_RED, linewidth=2, alpha=0.7)
    ax.plot(t_svd, label="Trained", color=ACCENT_BLUE, linewidth=2)
    ax.set_xlabel("Singular Value Index")
    ax.set_ylabel("Singular Value")
    ax.set_title(title)
    ax.set_yscale("log")
    ax.legend()
    ax.grid(alpha=0.2)
    
    # Compute effective rank (how many SVs needed for 90% of energy)
    t_cumsum = np.cumsum(t_svd**2) / np.sum(t_svd**2)
    r_cumsum = np.cumsum(r_svd**2) / np.sum(r_svd**2)
    t_rank90 = np.searchsorted(t_cumsum, 0.9) + 1
    r_rank90 = np.searchsorted(r_cumsum, 0.9) + 1
    ax.text(0.98, 0.5, f"90% energy rank:\nRandom: {r_rank90}\nTrained: {t_rank90}",
            transform=ax.transAxes, ha="right", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor=DARK_SURFACE, alpha=0.8))

plt.tight_layout()
plt.show()
print("Trained weights are approximately low-rank: a few directions dominate.")
print("Random weights have uniform singular values (full rank = no structure).")

## 5. Embedding Space: Random Blob → Semantic Clusters

The embedding layer maps each token to a vector. Before training, these vectors are random noise. After training, semantically similar tokens cluster together.

In [ ]:
from sklearn.decomposition import PCA

# Sample 2000 token embeddings for visualization
n_sample = 2000
torch.manual_seed(42)
indices = torch.randperm(trained_params["model.embed_tokens.weight"].shape[0])[:n_sample]

trained_embeds = trained_params["model.embed_tokens.weight"].data[indices].float().numpy()
random_embeds = random_params["model.embed_tokens.weight"].data[indices].float().numpy()

# PCA to 2D
pca_trained = PCA(n_components=2).fit_transform(trained_embeds)
pca_random = PCA(n_components=2).fit_transform(random_embeds)

# Categorize tokens for coloring
sample_tokens = [tokenizer.decode([idx.item()]) for idx in indices]
categories = []
for tok in sample_tokens:
    if tok.strip().isdigit():
        categories.append("Numbers")
    elif tok.strip() and all(c in "!@#$%^&*()[]{}|;:',.<>?/\\`~+-=_\"" for c in tok.strip()):
        categories.append("Punctuation")
    elif any('\u4e00' <= c <= '\u9fff' for c in tok):
        categories.append("Chinese")
    else:
        categories.append("Other")

cat_colors = {"Numbers": "#ef5350", "Punctuation": "#4fc3f7", "Chinese": "#81c784", "Other": "#555555"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for cat in ["Other", "Numbers", "Punctuation", "Chinese"]:
    mask = [c == cat for c in categories]
    ax1.scatter(pca_random[mask, 0], pca_random[mask, 1], c=cat_colors[cat],
                label=cat, s=5, alpha=0.4 if cat == "Other" else 0.8)
    ax2.scatter(pca_trained[mask, 0], pca_trained[mask, 1], c=cat_colors[cat],
                label=cat, s=5, alpha=0.4 if cat == "Other" else 0.8)

ax1.set_title("Random Init: Uniform Blob")
ax1.legend(markerscale=3)
ax2.set_title("Trained: Semantic Clusters Emerge")
ax2.legend(markerscale=3)
for ax in [ax1, ax2]:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()
print("Numbers cluster together. Punctuation clusters together. Structure emerges from chaos.")

## 6. Output Probabilities: Uniform → Confident

Before training, the model assigns roughly equal probability to every token. After training, it produces sharp, contextually appropriate distributions.

In [ ]:
# Compare output distributions
inputs = tokenizer("The capital of France is", return_tensors="pt")

with torch.no_grad():
    trained_logits = trained(**inputs).logits[0, -1].float()
    random_logits = random_model(**inputs).logits[0, -1].float()

trained_probs = torch.softmax(trained_logits, dim=-1)
random_probs = torch.softmax(random_logits, dim=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top-20 tokens for trained model
top_t = torch.topk(trained_probs, 20)
tokens_t = [tokenizer.decode([i]) for i in top_t.indices]
ax1.barh(range(20), top_t.values.numpy()[::-1], color=ACCENT_BLUE)
ax1.set_yticks(range(20))
ax1.set_yticklabels(tokens_t[::-1], fontsize=9)
ax1.set_title("Trained: Top 20 Predictions")
ax1.set_xlabel("Probability")

# Top-20 tokens for random model
top_r = torch.topk(random_probs, 20)
tokens_r = [tokenizer.decode([i]) for i in top_r.indices]
ax2.barh(range(20), top_r.values.numpy()[::-1], color=ACCENT_RED)
ax2.set_yticks(range(20))
ax2.set_yticklabels(tokens_r[::-1], fontsize=9)
ax2.set_title("Random: Top 20 Predictions")
ax2.set_xlabel("Probability")

plt.tight_layout()
plt.show()

print(f"Trained model entropy: {-(trained_probs * trained_probs.log()).sum():.2f} nats")
print(f"Random model entropy:  {-(random_probs * random_probs.log()).sum():.2f} nats")
print(f"Maximum entropy (uniform over {trained_probs.shape[0]:,} tokens): {np.log(trained_probs.shape[0]):.2f} nats")

## Summary

| Aspect | Before Training | After Training |
|--------|----------------|----------------|
| **Weights** | Gaussian noise (σ=0.02) | Structured, wider distributions |
| **Singular values** | Flat spectrum (full rank) | Heavy-tailed (low-rank structure) |
| **Embeddings** | Random blob | Semantic clusters |
| **Output distribution** | Near-uniform (~max entropy) | Sharp, contextually appropriate |
| **Generation** | Random gibberish | Coherent, knowledgeable text |

Training transforms ~494M random numbers into a structured system that encodes language, knowledge, and reasoning.